# Laboratorium terbuka: epidemiologi

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 7. Seluruh perhitungan memakai NumPy, SciPy, dan Matplotlib, dapat dijalankan secara luring, dan tidak memakai PPLANE, MATLAB, atau kode proprieter.

Eksperimen memeriksa penskalaan infeksi virus, ambang invasi, invariansi wilayah biologis SIR dan endemik, klasifikasi simpul–spiral, serta satu penutupan referensi MSEIR dengan populasi yang tidak konstan.

**ID unit:** O005-LEGA-V101-CH07  
**ID notebook:** O005-LEGA-V101-CH07-NB01  
**Lisensi notebook:** CC BY-NC-SA 4.0  
**Asal komponen:** pendamping komputasi baru berdasarkan persamaan dan soal Bab 7; bukan salinan kode aplikasi yang dirujuk sumber.

## Batas sumber dan koreksi berkeyakinan tinggi

- Bab mula-mula mengizinkan laju pemulihan $\beta\ge0$, tetapi kemudian mendefinisikan $\delta=\beta/\alpha>0$ dan $R_0=\alpha/\beta$. Dua definisi terakhir memerlukan asumsi tambahan $\beta>0$.
- Pada model endemik, $\eta+\delta=1$ membuat $P_2=P_1=(1,0)$ dan menghasilkan arah eigen nol. Kesetimbangan endemik dengan $i>0$ memerlukan syarat ketat $\eta+\delta<1$.
- Prosa Gambar 7.3 memberi $(\eta,\delta)=(0{,}2,0{,}1)$, sedangkan keterangannya memberi $(0{,}1,0{,}2)$. Keduanya menghasilkan spiral, tetapi titik tetap endemiknya berbeda.
- Soal MSEIR tidak menentukan alokasi bayi, hilangnya kekebalan pasif, insidensi, progresi laten, atau pemulihan. Sistem yang diuji di bawah adalah satu penutupan referensi dengan asumsi yang dinyatakan, bukan satu-satunya jawaban sumber.

## 1. Model virus: penskalaan, titik tetap, dan ambang

Model berdimensi adalah

$$\dot X=\lambda-\delta X-bVX,\qquad \dot Y=bVX-aY,\qquad \dot V=kY-\kappa V.$$

Penskalaan tercetak memakai $\theta=\delta t$, sedangkan Soal 1 meminta waktu alternatif $\tau=\lambda kbt/\delta^2$. Kita memakai skala keadaan yang sama pada keduanya agar perpindahan $X\to Y$ tetap seimbang.

In [ ]:
import platform

import numpy as np
import scipy
from scipy.integrate import solve_ivp
import matplotlib
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)

def selesaikan(fun, rentang, awal, args=(), max_step=0.1):
    hasil = solve_ivp(
        fun,
        rentang,
        np.asarray(awal, dtype=float),
        args=args,
        method="DOP853",
        rtol=1e-11,
        atol=1e-13,
        max_step=max_step,
    )
    assert hasil.success
    return hasil

def akhiri_gambar(fig):
    if "agg" in matplotlib.get_backend().lower():
        plt.close(fig)
    else:
        plt.show()

def virus_berdimensi(t, z, lam, delta_cell, b, a, k, kappa):
    X, Y, V = z
    return np.array([lam - delta_cell * X - b * V * X, b * V * X - a * Y, k * Y - kappa * V])

def virus_kanonis(t, z, zeta, eta, mu):
    x, y, v = z
    return np.array([zeta - x - v * x, v * x - eta * y, y - mu * v])

def virus_waktu_alternatif(t, z, rho, eta, mu):
    x, y, v = z
    return np.array([1.0 - rho * x * (1.0 + v), rho * (v * x - eta * y), rho * (y - mu * v)])

def jacobian_virus(x, y, v, eta, mu):
    return np.array([[-1.0 - v, 0.0, -x], [v, -eta, x], [0.0, 1.0, -mu]])

# Dimensi disimpan sebagai eksponen (sel, virion, waktu).
dim_lam = np.array([1, 0, -1])
dim_k = np.array([-1, 1, -1])
dim_b = np.array([0, -1, -1])
dim_t = np.array([0, 0, 1])
dim_delta = np.array([0, 0, -1])
assert np.array_equal(dim_lam + dim_k + dim_b + dim_t - 2 * dim_delta, np.zeros(3, dtype=int))

# Substitusi numerik memeriksa kedua penskalaan terhadap model berdimensi.
lam, delta_cell, b, a, k, kappa = 3.0, 1.2, 0.04, 0.8, 2.5, 0.7
X0 = delta_cell**2 / (k * b)
V0 = delta_cell / b
q_alt = lam * k * b / delta_cell**2
zeta = lam * k * b / delta_cell**3
eta = a / delta_cell
mu = kappa / delta_cell
rho = delta_cell**3 / (lam * k * b)
z_dim = np.array([1.7, 0.4, 2.2])
z_tak_dim = np.array([z_dim[0] / X0, z_dim[1] / X0, z_dim[2] / V0])
rhs_dim = virus_berdimensi(0.0, z_dim, lam, delta_cell, b, a, k, kappa)
rhs_theta_dari_dim = rhs_dim / np.array([X0 * delta_cell, X0 * delta_cell, V0 * delta_cell])
rhs_tau_dari_dim = rhs_dim / np.array([X0 * q_alt, X0 * q_alt, V0 * q_alt])
assert np.allclose(rhs_theta_dari_dim, virus_kanonis(0.0, z_tak_dim, zeta, eta, mu))
assert np.allclose(rhs_tau_dari_dim, virus_waktu_alternatif(0.0, z_tak_dim, rho, eta, mu))
assert np.isclose(rho, 1.0 / zeta)

def titik_virus(zeta, eta, mu):
    E0 = np.array([zeta, 0.0, 0.0])
    E_star = np.array([eta * mu, (zeta - eta * mu) / eta, (zeta - eta * mu) / (eta * mu)])
    return E0, E_star

def eigen_bebas_infeksi(zeta, eta, mu):
    return np.linalg.eigvals(jacobian_virus(zeta, 0.0, 0.0, eta, mu))

zeta_bawah, eta_bawah, mu_bawah = 0.6, 1.0, 0.8
zeta_atas, eta_atas, mu_atas = 1.2, 0.8, 0.7
E0_bawah, E_star_bawah = titik_virus(zeta_bawah, eta_bawah, mu_bawah)
E0_atas, E_star_atas = titik_virus(zeta_atas, eta_atas, mu_atas)
assert np.allclose(virus_kanonis(0.0, E0_bawah, zeta_bawah, eta_bawah, mu_bawah), 0.0)
assert np.allclose(virus_kanonis(0.0, E0_atas, zeta_atas, eta_atas, mu_atas), 0.0)
assert np.allclose(virus_kanonis(0.0, E_star_atas, zeta_atas, eta_atas, mu_atas), 0.0)
assert np.any(E_star_bawah < 0.0) and np.all(E_star_atas > 0.0)
eig_bawah = eigen_bebas_infeksi(zeta_bawah, eta_bawah, mu_bawah)
eig_atas = eigen_bebas_infeksi(zeta_atas, eta_atas, mu_atas)
assert np.all(eig_bawah.real < 0.0)
assert np.any(eig_atas.real > 0.0) and np.any(eig_atas.real < 0.0)

print(f"Python/NumPy/SciPy/Matplotlib: {platform.python_version()} / {np.__version__} / {scipy.__version__} / {matplotlib.__version__}")
print(f"Pemeriksaan waktu alternatif: zeta={zeta:.8f}, rho={rho:.8f}, rho*zeta={rho*zeta:.8f}")
print("Eigen bebas infeksi, bawah/atas ambang:", eig_bawah, eig_atas)
print("Titik endemik di atas ambang:", E_star_atas)

In [ ]:
awal_virus = np.array([0.7, 0.08, 0.06])
hasil_bawah = selesaikan(virus_kanonis, (0.0, 160.0), awal_virus, (zeta_bawah, eta_bawah, mu_bawah), max_step=0.08)
hasil_atas = selesaikan(virus_kanonis, (0.0, 140.0), awal_virus, (zeta_atas, eta_atas, mu_atas), max_step=0.08)
assert np.min(hasil_bawah.y) >= -1e-12 and np.min(hasil_atas.y) >= -1e-12
galat_bawah = float(np.linalg.norm(hasil_bawah.y[:, -1] - E0_bawah))
galat_atas = float(np.linalg.norm(hasil_atas.y[:, -1] - E_star_atas))
assert galat_bawah < 1e-8
assert galat_atas < 1e-7

fig, sumbu = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)
for indeks, nama in enumerate(("x: sel tak terinfeksi", "y: sel terinfeksi", "v: virus bebas")):
    sumbu[0].plot(hasil_bawah.t, hasil_bawah.y[indeks], label=nama)
    sumbu[1].plot(hasil_atas.t, hasil_atas.y[indeks], label=nama)
sumbu[0].set(title=r"$\mathcal R_v<1$: infeksi menghilang", xlabel=r"$\tau$", ylabel="keadaan tak berdimensi")
sumbu[1].set(title=r"$\mathcal R_v>1$: kesetimbangan endemik", xlabel=r"$\tau$", ylabel="keadaan tak berdimensi")
for ax in sumbu:
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
akhiri_gambar(fig)

print(f"R_v bawah/atas={zeta_bawah/(eta_bawah*mu_bawah):.8f} / {zeta_atas/(eta_atas*mu_atas):.8f}")
print(f"Galat akhir bebas-infeksi/endemik={galat_bawah:.3e} / {galat_atas:.3e}")

## 2. SIR klasik: segitiga dan invarian lintasan

Untuk $\delta=\beta/\alpha>0$,

$$s'=-si,\qquad i'=i(s-\delta),\qquad \mathcal T=\{s\ge0,\ i\ge0,\ s+i\le1\}.$$

Selain invariansi segitiga, lintasan interior mempertahankan $H(s,i)=s+i-\delta\log s$. Pemeriksaan ini lebih kuat daripada sekadar melihat grafik.

In [ ]:
def sir_klasik(t, z, delta):
    s, i = z
    return np.array([-s * i, i * (s - delta)])

delta_sir = 0.2
assert sir_klasik(0.0, [0.0, 0.4], delta_sir)[0] == 0.0
assert sir_klasik(0.0, [0.4, 0.0], delta_sir)[1] == 0.0
for s_batas in (0.0, 0.2, 0.7, 1.0):
    i_batas = 1.0 - s_batas
    assert np.isclose(np.sum(sir_klasik(0.0, [s_batas, i_batas], delta_sir)), -delta_sir * i_batas)

awal_sir = ([0.99, 0.01], [0.80, 0.15], [0.55, 0.35], [0.25, 0.60])
hasil_sir = [selesaikan(sir_klasik, (0.0, 180.0), z0, (delta_sir,), max_step=0.08) for z0 in awal_sir]
drift_invarian = []
for hasil in hasil_sir:
    s, i = hasil.y
    assert np.min(s) > 0.0 and np.min(i) >= -1e-12
    assert np.max(s + i) <= 1.0 + 1e-11
    H = s + i - delta_sir * np.log(s)
    drift_invarian.append(float(np.max(np.abs(H - H[0]))))
    assert i[-1] < 1e-10
assert max(drift_invarian) < 1e-9

s_grid = np.linspace(0.0, 1.0, 25)
i_grid = np.linspace(0.0, 1.0, 25)
S, I = np.meshgrid(s_grid, i_grid)
U = -S * I
W = I * (S - delta_sir)
mask = S + I > 1.0
U = np.ma.array(U, mask=mask)
W = np.ma.array(W, mask=mask)
norma = np.ma.sqrt(U * U + W * W)
norma = np.ma.where(norma == 0.0, 1.0, norma)
fig, ax = plt.subplots(figsize=(6.5, 5.4), constrained_layout=True)
ax.quiver(S, I, U / norma, W / norma, color="#AAAAAA", alpha=0.8)
for hasil in hasil_sir:
    ax.plot(hasil.y[0], hasil.y[1], linewidth=1.7)
ax.plot([0, 1], [1, 0], "k--", linewidth=1.0, label="$s+i=1$")
ax.axvline(delta_sir, color="#D55E00", linestyle=":", label=r"$s=\delta$")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="$s$", ylabel="$i$", title=r"Bidang fase SIR klasik, $\delta=0{,}2$")
ax.legend()
ax.grid(alpha=0.18)
akhiri_gambar(fig)

print("SIR s akhir:", np.array([h.y[0, -1] for h in hasil_sir]))
print(f"Drift maksimum H={max(drift_invarian):.3e}; i akhir maksimum={max(h.y[1,-1] for h in hasil_sir):.3e}")

## 3. SIR endemik: invariansi dan dua rezim bidang fase

$$s'=\eta(1-s)-si,\qquad i'=i(s-\eta-\delta).$$

Pada sisi miring $s+i=1$, berlaku $(s+i)'=-\delta i\le0$. Titik bebas penyakit adalah $P_1=(1,0)$; titik endemik $P_2$ mempunyai koordinat positif hanya jika $\eta+\delta<1$.

In [ ]:
def sir_endemik(t, z, eta, delta):
    s, i = z
    return np.array([eta * (1.0 - s) - s * i, i * (s - eta - delta)])

def jacobian_endemik(s, i, eta, delta):
    return np.array([[-eta - i, -s], [i, s - eta - delta]])

def titik_endemik(eta, delta):
    q = eta + delta
    return np.array([1.0, 0.0]), np.array([q, eta * (1.0 - q) / q])

for eta_uji, delta_uji in ((1.0, 0.2), (0.1, 0.2)):
    assert sir_endemik(0.0, [0.0, 0.4], eta_uji, delta_uji)[0] > 0.0
    assert sir_endemik(0.0, [0.4, 0.0], eta_uji, delta_uji)[1] == 0.0
    for s_batas in (0.0, 0.2, 0.7, 1.0):
        i_batas = 1.0 - s_batas
        assert np.isclose(np.sum(sir_endemik(0.0, [s_batas, i_batas], eta_uji, delta_uji)), -delta_uji * i_batas)

kasus_endemik = ((1.0, 0.2, 100.0, "bebas penyakit"), (0.1, 0.2, 220.0, "endemik"))
awal_endemik = ([0.85, 0.10], [0.55, 0.30], [0.20, 0.60])
hasil_per_kasus = []
fig, sumbu = plt.subplots(1, 2, figsize=(11.3, 4.9), constrained_layout=True)
for ax, (eta_k, delta_k, akhir, nama) in zip(sumbu, kasus_endemik):
    P1, P2 = titik_endemik(eta_k, delta_k)
    target = P1 if eta_k + delta_k > 1.0 else P2
    assert np.allclose(sir_endemik(0.0, P1, eta_k, delta_k), 0.0)
    assert np.allclose(sir_endemik(0.0, P2, eta_k, delta_k), 0.0)
    hasil_kasus = [selesaikan(sir_endemik, (0.0, akhir), z0, (eta_k, delta_k), max_step=0.08) for z0 in awal_endemik]
    hasil_per_kasus.append(hasil_kasus)
    s_grid = np.linspace(0.0, 1.0, 24)
    i_grid = np.linspace(0.0, 1.0, 24)
    S, I = np.meshgrid(s_grid, i_grid)
    U = eta_k * (1.0 - S) - S * I
    W = I * (S - eta_k - delta_k)
    mask = S + I > 1.0
    U, W = np.ma.array(U, mask=mask), np.ma.array(W, mask=mask)
    norma = np.ma.sqrt(U * U + W * W)
    norma = np.ma.where(norma == 0.0, 1.0, norma)
    ax.quiver(S, I, U / norma, W / norma, color="#AAAAAA", alpha=0.8)
    for hasil in hasil_kasus:
        assert np.min(hasil.y) >= -1e-12 and np.max(np.sum(hasil.y, axis=0)) <= 1.0 + 1e-11
        ax.plot(hasil.y[0], hasil.y[1], linewidth=1.6)
    ax.plot(*target, "o", color="#D55E00", label=f"target {nama}")
    ax.plot([0, 1], [1, 0], "k--", linewidth=1.0)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="$s$", ylabel="$i$", title=f"$\\eta={eta_k:g}$, $\\delta={delta_k:g}$")
    ax.legend()
    ax.grid(alpha=0.18)
akhiri_gambar(fig)

P1_bebas, _ = titik_endemik(1.0, 0.2)
_, P2_endemik = titik_endemik(0.1, 0.2)
galat_bebas = max(np.linalg.norm(h.y[:, -1] - P1_bebas) for h in hasil_per_kasus[0])
galat_endemik = max(np.linalg.norm(h.y[:, -1] - P2_endemik) for h in hasil_per_kasus[1])
assert galat_bebas < 1e-8 and galat_endemik < 1e-7
eig_P1 = np.linalg.eigvals(jacobian_endemik(*P1_bebas, 1.0, 0.2))
eig_P2 = np.linalg.eigvals(jacobian_endemik(*P2_endemik, 0.1, 0.2))
assert np.all(eig_P1 < 0.0)
assert np.all(eig_P2.real < 0.0) and np.any(np.abs(eig_P2.imag) > 0.0)

print("Endemik P2 untuk keterangan Gambar 7.3 (eta=0,1; delta=0,2):", P2_endemik)
print("Eigen P1 bebas penyakit / P2 endemik:", eig_P1, eig_P2)
print(f"Galat maksimum dua rezim={galat_bebas:.3e} / {galat_endemik:.3e}")

## 4. Soal 4: permukaan parameter simpul–spiral

Dengan $q=\eta+\delta<1$, jejak dan determinan di $P_2$ adalah $T=-\eta/q$ dan $D=\eta(1-q)$. Batas tipe linear ditentukan oleh

$$\Delta=T^2-4D=\frac{\eta^2}{q^2}-4\eta(1-q).$$

Nilai $\Delta\ge0$ memberi simpul stabil; $\Delta<0$ memberi spiral stabil.

In [ ]:
def diskriminan_endemik(eta, delta):
    q = eta + delta
    return eta**2 / q**2 - 4.0 * eta * (1.0 - q)

eta_node, delta_node = 0.15, 0.05
eta_spiral, delta_spiral = 0.10, 0.20
P1_node, P2_node = titik_endemik(eta_node, delta_node)
P1_spiral, P2_spiral = titik_endemik(eta_spiral, delta_spiral)
eig_node = np.linalg.eigvals(jacobian_endemik(*P2_node, eta_node, delta_node))
eig_spiral = np.linalg.eigvals(jacobian_endemik(*P2_spiral, eta_spiral, delta_spiral))
Delta_node = diskriminan_endemik(eta_node, delta_node)
Delta_spiral = diskriminan_endemik(eta_spiral, delta_spiral)
assert np.isclose(Delta_node, 0.0825) and Delta_node > 0.0
assert Delta_spiral < 0.0
assert np.all(np.isreal(eig_node)) and np.all(eig_node < 0.0)
assert np.all(eig_spiral.real < 0.0) and np.any(np.abs(eig_spiral.imag) > 0.0)
q_node = eta_node + delta_node
assert eta_node >= 4.0 * q_node**2 * (1.0 - q_node)

eta_grid = np.linspace(0.005, 0.99, 260)
delta_grid = np.linspace(0.005, 0.99, 260)
ETA, DELTA = np.meshgrid(eta_grid, delta_grid)
Q = ETA + DELTA
DISK = ETA**2 / Q**2 - 4.0 * ETA * (1.0 - Q)
DISK = np.ma.array(DISK, mask=Q >= 1.0)
assert np.any(DISK.compressed() > 0.0) and np.any(DISK.compressed() < 0.0)

fig, ax = plt.subplots(figsize=(7.0, 5.5), constrained_layout=True)
warna = ax.contourf(ETA, DELTA, DISK, levels=[-2.0, 0.0, 2.0], colors=["#56B4E9", "#E69F00"], alpha=0.75)
ax.contour(ETA, DELTA, DISK, levels=[0.0], colors="black", linewidths=1.4)
ax.plot(eta_node, delta_node, "o", color="#009E73", label="contoh simpul")
ax.plot(eta_spiral, delta_spiral, "s", color="#CC79A7", label="contoh spiral")
ax.plot([0, 1], [1, 0], "k--", linewidth=1.0, label=r"$\eta+\delta=1$")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r"$\eta$", ylabel=r"$\delta$", title="Tipe kesetimbangan endemik di bidang parameter")
ax.legend()
ax.grid(alpha=0.18)
akhiri_gambar(fig)

print(f"Contoh simpul: Delta={Delta_node:.8f}, eigen={eig_node}")
print(f"Contoh spiral: Delta={Delta_spiral:.8f}, eigen={eig_spiral}")

## 5. Soal 5: satu penutupan referensi MSEIR

Sumber tidak menentukan hukum transisi, jadi kita menyatakan satu pilihan: fraksi $p$ bayi masuk kelas kekebalan pasif $M$; kekebalan itu hilang dengan laju $\omega$; insidensi bersifat frekuensi-tergantung $\alpha SI/N$; progresi $E\to I$ berlaju $\sigma$; pemulihan $I\to R$ berlaju $\gamma$; dan kematian alami $\mu$ bekerja pada semua kelas hidup. Kekebalan setelah pulih bersifat permanen dan tidak ada kematian akibat penyakit.

Dengan asumsi tersebut, penjumlahan lima persamaan harus memberi $N'=(\nu-\mu)N$.

In [ ]:
def mseir(t, z, nu, mu, p, omega, alpha, sigma, gamma):
    M, S, E, I, R = z
    N = M + S + E + I + R
    insidensi = alpha * S * I / N if N > 0.0 else 0.0
    return np.array([
        p * nu * N - (omega + mu) * M,
        (1.0 - p) * nu * N + omega * M - insidensi - mu * S,
        insidensi - (sigma + mu) * E,
        sigma * E - (gamma + mu) * I,
        gamma * I - mu * R,
    ])

parameter_mseir = (0.025, 0.015, 0.8, 0.35, 2.0, 0.6, 0.4)
nu_m, mu_m, p_m, omega_m, alpha_m, sigma_m, gamma_m = parameter_mseir
z_uji = np.array([0.07, 0.72, 0.09, 0.08, 0.04])
rhs_uji = mseir(0.0, z_uji, *parameter_mseir)
assert np.isclose(np.sum(rhs_uji), (nu_m - mu_m) * np.sum(z_uji))

# Pada setiap muka ortan, komponen yang nol mempunyai turunan tidak negatif.
for indeks in range(5):
    z_batas = z_uji.copy()
    z_batas[indeks] = 0.0
    assert mseir(0.0, z_batas, *parameter_mseir)[indeks] >= 0.0

awal_mseir = np.array([0.05, 0.86, 0.05, 0.04, 0.0])
hasil_mseir = selesaikan(mseir, (0.0, 60.0), awal_mseir, parameter_mseir, max_step=0.04)
hasil_mseir_ulang = selesaikan(mseir, (0.0, 60.0), awal_mseir, parameter_mseir, max_step=0.04)
assert np.array_equal(hasil_mseir.t, hasil_mseir_ulang.t)
assert np.array_equal(hasil_mseir.y, hasil_mseir_ulang.y)
assert np.min(hasil_mseir.y) >= -1e-12
N_numerik = np.sum(hasil_mseir.y, axis=0)
N_eksak = np.sum(awal_mseir) * np.exp((nu_m - mu_m) * hasil_mseir.t)
galat_N = float(np.max(np.abs(N_numerik - N_eksak)))
assert galat_N < 1e-9

fig, ax = plt.subplots(figsize=(8.4, 4.9), constrained_layout=True)
for indeks, nama in enumerate(("M", "S", "E", "I", "R")):
    ax.plot(hasil_mseir.t, hasil_mseir.y[indeks], label=nama)
ax.plot(hasil_mseir.t, N_numerik, "k--", linewidth=1.4, label="N total")
ax.set(xlabel="waktu", ylabel="jumlah relatif", title=r"Penutupan referensi MSEIR dengan $\nu\ne\mu$")
ax.legend(ncol=3)
ax.grid(alpha=0.2)
akhiri_gambar(fig)

print(f"MSEIR N(60) numerik/eksak={N_numerik[-1]:.8f} / {N_eksak[-1]:.8f}; galat maksimum={galat_N:.3e}")
print("MSEIR keadaan akhir (M,S,E,I,R):", hasil_mseir.y[:, -1])

## Kesimpulan

Notebook ini memeriksa penskalaan virus langsung terhadap persamaan berdimensi, membuktikan ambang invasi melalui nilai eigen, mengukur invarian SIR, menguji semua sisi segitiga biologis, menghasilkan bidang fase endemik tanpa perangkat proprieter, memetakan permukaan parameter simpul–spiral, dan memverifikasi keseimbangan populasi MSEIR dengan $\nu\ne\mu$. Semua parameter demonstrasi dinyatakan; sistem MSEIR tetap diberi label sebagai penutupan referensi tambahan.